# Advanced Machine Learning (21CSE425T) — Unit 1: Reinforcement Learning

Standalone Colab notebook for **T1–T3**: RL chatbot (Q-learning), Deep Q-Learning (DQN), and Policy Gradient (REINFORCE). Run top-to-bottom; the setup cell installs all dependencies.

---
# Unit 1 — Reinforcement Learning

Installs Gymnasium + PyTorch (+ optional Stable-Baselines3) and implements three canonical RL methods:
a **tabular Q-learning chatbot** (T1), **Deep Q-Learning** (T2), and **REINFORCE policy gradient** (T3).

### Unit 1 — Setup & installs

In [ ]:
# Install RL dependencies. torch is pre-installed on Colab; gymnasium/sb3 are not.
# (Quiet install; re-running is harmless.)
!pip -q install "gymnasium==0.29.1" "stable-baselines3==2.3.2" 2>/dev/null

# torch ships with Colab. If running locally without it, uncomment the next line:
# !pip -q install torch

import sys, platform
import numpy as np
import torch
import gymnasium as gym

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", torch.__version__)
print("Gymnasium   :", gym.__version__)
print("CUDA avail. :", torch.cuda.is_available())

# One global device handle reused by every PyTorch task in this notebook.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## T1 — Reinforcement Learning chatbot (tabular Q-learning)

**Idea.** We frame a tiny intent-driven chatbot as an RL problem. The *state* is the user's
**intent** (greeting, thanks, help, …). The *action* is **which reply the bot picks**. The bot gets
`+1` reward when its reply matches the intent and `-1` otherwise. With **Q-learning** the agent learns
the optimal state→action (intent→reply) policy purely from reward feedback — no labelled (intent,reply)
supervision. At inference we classify the user's text to an intent (bag-of-words nearest match) and let
the learned Q-table choose the reply.

Swap `INTENT_DATA` for any public intent dataset (e.g. Kaggle *Chatbot intents* / `intents.json`) and the
same loop applies.

In [ ]:
import numpy as np, random
np.random.seed(0); random.seed(0)

# ---- "Public dataset" stand-in: intents -> example utterances (swap for intents.json) ----
INTENT_DATA = {
    "greeting": ["hi", "hello", "hey there", "good morning", "howdy"],
    "thanks":   ["thanks", "thank you", "much appreciated", "thanks a lot"],
    "help":     ["can you help", "i need help", "support please", "help me"],
    "bye":      ["bye", "goodbye", "see you", "catch you later"],
    "weather":  ["what is the weather", "is it raining", "weather today", "forecast"],
}
# Each intent has one "correct" reply category (the action the agent must learn to choose).
REPLIES = {
    "greeting": "Hello! How can I help you today?",
    "thanks":   "You're welcome! Happy to help.",
    "help":     "Sure — tell me what you need help with.",
    "bye":      "Goodbye! Have a great day.",
    "weather":  "I can't fetch live weather, but it's always sunny in here.",
}
INTENTS = list(INTENT_DATA.keys())              # state space
ACTIONS = list(REPLIES.keys())                  # action space (one reply per intent)
n_states, n_actions = len(INTENTS), len(ACTIONS)

# ---- Simple bag-of-words intent classifier (used only at inference time) ----
VOCAB = sorted({w for phrases in INTENT_DATA.values() for p in phrases for w in p.split()})
def vectorize(text):
    v = np.zeros(len(VOCAB))
    for w in text.lower().split():
        if w in VOCAB: v[VOCAB.index(w)] += 1
    return v
# Pre-compute a prototype vector per intent (mean of its example utterances).
PROTO = {it: np.mean([vectorize(p) for p in INTENT_DATA[it]], axis=0) for it in INTENTS}
def classify_intent(text):
    v = vectorize(text)
    # cosine-like nearest prototype
    scores = {it: float(v @ PROTO[it] / (np.linalg.norm(v)*np.linalg.norm(PROTO[it]) + 1e-9))
              for it in INTENTS}
    return max(scores, key=scores.get)

# ---- Q-learning over the (intent -> reply) decision ----
Q = np.zeros((n_states, n_actions))
alpha, gamma, eps = 0.5, 0.9, 0.2          # learning rate, discount, exploration
EPISODES = 4000

def reward(state_idx, action_idx):
    return 1.0 if INTENTS[state_idx] == ACTIONS[action_idx] else -1.0

for ep in range(EPISODES):
    s = random.randrange(n_states)                      # environment presents a random intent
    a = (random.randrange(n_actions) if random.random() < eps  # epsilon-greedy
         else int(np.argmax(Q[s])))
    r = reward(s, a)
    # Single-step bandit-style episode: next state is terminal, so target = r.
    Q[s, a] += alpha * (r + gamma * 0.0 - Q[s, a])

learned_policy = {INTENTS[s]: ACTIONS[int(np.argmax(Q[s]))] for s in range(n_states)}
print("Learned intent -> reply policy:")
for k, v in learned_policy.items():
    print(f"  {k:9s} -> {v}")
print("\nPolicy correct for all intents:",
      all(k == v for k, v in learned_policy.items()))

In [ ]:
# ---- Chat with the trained agent ----
def chatbot_reply(text):
    intent = classify_intent(text)                 # text -> state
    action = int(np.argmax(Q[INTENTS.index(intent)]))  # state -> best action via Q-table
    return intent, REPLIES[ACTIONS[action]]

for msg in ["hello", "thanks a lot", "i need help", "what is the weather", "see you"]:
    intent, reply = chatbot_reply(msg)
    print(f"User: {msg}\n  [intent={intent}] Bot: {reply}\n")

## T2 — Deep Q-Learning (DQN) with Gymnasium + PyTorch

A from-scratch DQN on **CartPole-v1** with the three ingredients that stabilise deep RL:
an **experience replay buffer**, a **target network**, and an **ε-greedy** exploration schedule.

In [ ]:
import gymnasium as gym
import numpy as np, random, math
from collections import deque, namedtuple
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)

# ---- Q-network: maps a state vector to one Q-value per action ----
class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)

# ---- Experience replay: decorrelates samples for stable training ----
Transition = namedtuple("Transition", ("s", "a", "r", "s2", "done"))
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, batch_size):
        batch = Transition(*zip(*random.sample(self.buf, batch_size)))
        return batch
    def __len__(self):
        return len(self.buf)

In [ ]:
def train_dqn(episodes=300, batch_size=64, gamma=0.99,
              eps_start=1.0, eps_end=0.05, eps_decay=0.995,
              target_sync=10, lr=1e-3, solve_threshold=475.0):
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    policy_net = QNetwork(obs_dim, n_actions).to(DEVICE)
    target_net = QNetwork(obs_dim, n_actions).to(DEVICE)
    target_net.load_state_dict(policy_net.state_dict())   # start identical
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    buffer = ReplayBuffer()
    eps = eps_start
    episode_rewards = []

    for ep in range(episodes):
        state, _ = env.reset(seed=ep)
        done, total = False, 0.0
        while not done:
            # epsilon-greedy action selection
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    s = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
                    action = int(policy_net(s).argmax().item())

            nxt, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buffer.push(state, action, reward, nxt, float(done))
            state = nxt
            total += reward

            # ---- learn once we have enough samples ----
            if len(buffer) >= batch_size:
                b = buffer.sample(batch_size)
                s  = torch.as_tensor(np.array(b.s),  dtype=torch.float32, device=DEVICE)
                a  = torch.as_tensor(b.a, dtype=torch.int64, device=DEVICE).unsqueeze(1)
                r  = torch.as_tensor(b.r, dtype=torch.float32, device=DEVICE).unsqueeze(1)
                s2 = torch.as_tensor(np.array(b.s2), dtype=torch.float32, device=DEVICE)
                d  = torch.as_tensor(b.done, dtype=torch.float32, device=DEVICE).unsqueeze(1)

                q_sa = policy_net(s).gather(1, a)                        # Q(s,a)
                with torch.no_grad():
                    q_next = target_net(s2).max(1, keepdim=True)[0]      # max_a' Q_target(s',a')
                    target = r + gamma * q_next * (1.0 - d)              # TD target
                loss = nn.functional.smooth_l1_loss(q_sa, target)       # Huber loss

                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_value_(policy_net.parameters(), 100.0)
                optimizer.step()

        eps = max(eps_end, eps * eps_decay)                 # decay exploration
        if ep % target_sync == 0:                           # periodically sync target net
            target_net.load_state_dict(policy_net.state_dict())
        episode_rewards.append(total)

        avg = np.mean(episode_rewards[-20:])
        if ep % 20 == 0:
            print(f"ep {ep:3d} | reward {total:6.1f} | avg20 {avg:6.1f} | eps {eps:.3f}")
        if len(episode_rewards) >= 20 and avg >= solve_threshold:
            print(f"Solved at episode {ep} (avg20={avg:.1f}).")
            break

    env.close()
    return policy_net, episode_rewards

dqn_net, dqn_rewards = train_dqn(episodes=300)

In [ ]:
# ---- Learning curve ----
plt.figure(figsize=(8,4))
plt.plot(dqn_rewards, alpha=0.4, label="episode reward")
if len(dqn_rewards) >= 20:
    ma = np.convolve(dqn_rewards, np.ones(20)/20, mode="valid")
    plt.plot(range(19, 19+len(ma)), ma, label="20-episode moving avg")
plt.xlabel("Episode"); plt.ylabel("Total reward"); plt.title("T2 — DQN on CartPole-v1")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## T3 — Policy Gradient (REINFORCE)

The **REINFORCE** algorithm (Williams, 1992) optimises a stochastic policy directly by gradient ascent on
expected return: the loss is `-Σ logπ(aₜ|sₜ) · Gₜ`, where `Gₜ` is the discounted return (we standardise
returns to reduce variance). Same CartPole environment, a very different learning principle from DQN.

In [ ]:
import gymnasium as gym
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
from torch.distributions import Categorical
import matplotlib.pyplot as plt

torch.manual_seed(1); np.random.seed(1)

class PolicyNetwork(nn.Module):
    """Outputs action-probability logits for a given state."""
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return self.net(x)

def train_reinforce(episodes=600, gamma=0.99, lr=1e-3, solve_threshold=475.0):
    env = gym.make("CartPole-v1")
    policy = PolicyNetwork(env.observation_space.shape[0], env.action_space.n).to(DEVICE)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    episode_rewards = []

    for ep in range(episodes):
        state, _ = env.reset(seed=ep)
        log_probs, rewards = [], []
        done = False
        while not done:
            s = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
            dist = Categorical(logits=policy(s))   # stochastic policy
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            state, reward, terminated, truncated, _ = env.step(int(action.item()))
            rewards.append(reward)
            done = terminated or truncated

        # ---- discounted returns G_t, then standardise ----
        returns, G = [], 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.as_tensor(returns, dtype=torch.float32, device=DEVICE)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # ---- policy-gradient loss ----
        loss = torch.stack([-lp * G for lp, G in zip(log_probs, returns)]).sum()
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        episode_rewards.append(sum(rewards))
        avg = np.mean(episode_rewards[-20:])
        if ep % 25 == 0:
            print(f"ep {ep:3d} | reward {sum(rewards):6.1f} | avg20 {avg:6.1f}")
        if len(episode_rewards) >= 20 and avg >= solve_threshold:
            print(f"Solved at episode {ep} (avg20={avg:.1f}).")
            break

    env.close()
    return policy, episode_rewards

pg_policy, pg_rewards = train_reinforce(episodes=600)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(pg_rewards, alpha=0.4, label="episode reward")
if len(pg_rewards) >= 20:
    ma = np.convolve(pg_rewards, np.ones(20)/20, mode="valid")
    plt.plot(range(19, 19+len(ma)), ma, label="20-episode moving avg")
plt.xlabel("Episode"); plt.ylabel("Total reward"); plt.title("T3 — REINFORCE on CartPole-v1")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()